# Part 3 — Language Modeling

> Part 3 of **[Tokens to Agents](../../README.md)** — the foundations of agentic AI, built from scratch.

The word *jumped* appears seven times in training. Part 1's count-based bigram hedges on what
follows it; the model built here says *to* with probability 0.37, because it learned that *jumped*
behaves like *rose* and borrowed *rose*'s seventy-eight sightings. This notebook walks the same
experiment as `run.py` in narrative order: one SGD step on paper, the gradient check, the final
model, the borrowed row, and the comparison with counting.

In [ ]:
import numpy as np
from lm import (
    Params,
    backward,
    bigrams,
    close_vocabulary,
    forward,
    gradient_check,
    perplexity,
    sgd_step,
    train,
)
from lm.model import loss
from run import BORROW, CFG, PROBES, borrowed_rows, nearest, part1_baselines, toy_problem

from core import viz
from core.corpus import sentences, split

viz.use_theme()
train_s, test_s = split(sentences())
vocab = close_vocabulary(train_s)
fit_s, valid_s = split(train_s, test_frac=0.1, seed=0)
fit, valid, test = (bigrams(s, vocab) for s in (fit_s, valid_s, test_s))
print(f'{len(fit):,} training pairs · {len(valid):,} validation · {len(test):,} test · vocab {vocab.size:,}')

## One SGD step, on paper and in code

Three words, two dimensions, previous token 1, target token 2. The forward pass is
`h = E[x]`, `z = W h + b`, `p = softmax(z)`; the loss is `−log p[y]`. The only derivative that
carries anything is `∂L/∂z = p − onehot(y)` — every other gradient is that vector routed back
through the two matrix products. With learning rate 0.1 the looked-up embedding should move from
`[0.000, 0.100]` to `[0.0017, 0.1017]`.

In [ ]:
params = Params(
    E=np.array([[0.10, 0.00], [0.00, 0.10], [0.10, 0.10], [0.00, 0.00]]),  # last row is <s>
    W=np.array([[0.05, 0.00], [0.00, 0.05], [0.05, 0.05]]),
    b=np.zeros(3),
)
x, y = np.array([1]), np.array([2])
h, p = forward(params, x)
grads = backward(params, x, y, h, p)
print('p          ', p.round(4), ' loss', round(loss(p, y), 3))
print('p − onehot ', (p - np.eye(3)[y]).round(4))
sgd_step(params, grads, lr=0.1)
print('E[1] after ', params.E[1].round(4))

## Is the derivation right?

Central finite differences nudge every parameter of a random toy model by ±1e-5 and compare the
measured slope with the analytic gradient. A correct derivative agrees to ~1e-8; a sign error
anywhere shows up as a disagreement of order one.

In [ ]:
print(f'max relative error: {gradient_check(*toy_problem()):.1e}')

## Train the model

Plain mini-batch SGD — no momentum, no schedule — with validation perplexity recorded after every
epoch. The configuration is the winner of the sweeps in `run.py` (learning rate 5, width 64);
epoch 0 is the uniform guess, so the curve starts at the vocabulary size.

In [ ]:
params, hist = train(fit, valid, vocab.size, **CFG)
fig = viz.trend(
    np.arange(CFG['epochs'] + 1), {f"lr {CFG['lr']:g}": [vocab.size, *hist.valid_ppl]},
    title=f'From a uniform guess to perplexity {min(hist.valid_ppl):,.0f} in {CFG["epochs"]} epochs of hand-written SGD',
    xlabel='Epoch', ylabel='Validation perplexity', log=True,
)
ppl_test = perplexity(params, test)
print(f'best epoch {hist.best_epoch + 1} · test perplexity {ppl_test:,.1f}')

## Against Part 1's counting models

Same 970 test sentences, same vocabulary closure. The count bigram and the neural bigram see
exactly the same input — the previous word — so the gap between them is the value of learning a
factorised table instead of filling a sparse one.

In [ ]:
baselines = part1_baselines()
fig = viz.bars(
    [*baselines, f"Neural bigram (d={CFG['dim']})"], [*baselines.values(), ppl_test],
    title=f'Same context, no counts: neural bigram {ppl_test:,.0f} vs count bigram '
          f'{next(v for k, v in baselines.items() if k.startswith("Count bigram")):,.0f}',
    xlabel='Test perplexity (lower is better)', best='min', fmt='{:,.1f}',
)

## The row the model never saw

*rose* is followed by *to* or *by* seventy-eight times in training; *jumped* appears seven times
in total. Part 1's add-α estimator, recomputed here on the same pairs, spreads *jumped*'s mass
thinly over 4,495 words. The neural model has pushed *jumped* next to *rose* in the embedding
table, so it inherits *rose*'s row without ever earning it.

In [ ]:
borrow = borrowed_rows(params, fit, vocab)
common, rare = BORROW['contexts']
labels = [f'{ctx} → {n}' for ctx in BORROW['contexts'] for n in BORROW['nexts']]
fig = viz.paired_bars(
    labels,
    {'Neural bigram': [borrow[c]['p'][n][1] for c in BORROW['contexts'] for n in BORROW['nexts']],
     f"Count bigram (add-{BORROW['alpha']:g})": [borrow[c]['p'][n][0] for c in BORROW['contexts'] for n in BORROW['nexts']]},
    title=f"Seen {borrow[rare]['seen']} times, '{rare}' gets a confident prediction — "
          f"the model learned it behaves like '{common}' (seen {borrow[common]['seen']} times)",
    xlabel='P(next word | previous word)',
)

## The embedding table learned a word map for free

`E` was trained only to predict the next word. Ask it for nearest neighbours anyway and it
reproduces Part 2's geometry — profit-and-loss terms cluster, movement verbs cluster — because
both objectives are ultimately fed by what appears next to what.

In [ ]:
for w in PROBES:
    print(f'{w:<7}', ', '.join(f'{n} {s:.2f}' for n, s in nearest(params, vocab, w)))

## Next: transformers

This model reads one token and forgets the rest. Part 4 replaces the single lookup with attention —
every position reading every other — built from the same parts used here: an embedding table,
matrix products, a softmax.